# Hybrid Stage 1: XGBoost + Isolation Forest Router

Stage 1 combines two independent signals:

- **Supervised risk:** calibrated probability of `Attack` from XGBoost using the 41 original features and model configuration from `Experiment.ipynb`.
- **Novelty risk:** Isolation Forest percentile using only the correlation-selected combined features logged by `feature_selection_Anamoly_Detection.ipynb`.

The routing design follows `hybrid_xgb_iforest_router.py`. An external test set is held out first and used only for final evaluation. The remaining training data is divided into model-fit, trusted-normal reference, and routing-validation partitions. Thresholds are learned from explicit error budgets on the routing-validation partition—not selected on the test set.

## 1. Imports and experiment configuration

The threshold budgets below are operational requirements. Lower values make automatic decisions more conservative and usually increase the review queue.

In [1]:
import json
import platform
import sys
import tempfile
import time
from dataclasses import asdict
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import Markdown, display
from mlflow.models import infer_signature
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "Notebooks"))

from configs.config import EXPERIMENT_NAME, RANDOM_STATE
from hybrid_xgb_iforest_router import (
    HybridRouter,
    NormalNoveltyPercentile,
    learn_routing_thresholds,
    raw_iforest_anomaly_score,
    split_training_for_hybrid,
)

TRACKING_DB = (PROJECT_ROOT / "Notebooks" / "mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{TRACKING_DB.as_posix()}")
mlflow.set_experiment(EXPERIMENT_NAME)

RUN_NAME = "Hybrid_Stage1_XGBoost_IsolationForest"
MODEL_VERSION = "stage1-router-v1"
THRESHOLD_VERSION = "routing-thresholds-v1"
SOURCE_RUN_NAME = "combined_correlationfiltering_IsolationForest"
SOURCE_ARTIFACT = "feature_selection/selected_features_and_details.json"

REFERENCE_FRACTION = 0.15
VALIDATION_FRACTION = 0.15
CALIBRATION_FOLDS = 5
NORMAL_FIT_LIMIT = 6000

# Validation constraints used to learn the four routing thresholds.
THRESHOLD_BUDGETS = {
    "xgb_auto_attack_fpr": 0.001,
    "xgb_auto_normal_attack_escape_rate": 0.001,
    "xgb_auto_normal_region_attack_rate": 0.001,
    "novelty_high_fpr": 0.01,
    "novelty_extreme_fpr": 0.001,
}

print("MLflow tracking:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)
print("Threshold budgets:", THRESHOLD_BUDGETS)

MLflow tracking: sqlite:///D:/E Drive/Sentiflow-Network Intrusion Detection/Notebooks/mlflow.db
Experiment: Network_Intrusion_Detection
Threshold budgets: {'xgb_auto_attack_fpr': 0.001, 'xgb_auto_normal_attack_escape_rate': 0.001, 'xgb_auto_normal_region_attack_rate': 0.001, 'novelty_high_fpr': 0.01, 'novelty_extreme_fpr': 0.001}


## 2. Load the data and reserve the external test set

In [2]:
DATA_PATH = PROJECT_ROOT / "Data" / "Consolidated_df.csv"

ORIGINAL_FEATURES = (
    "duration", "protocoltype", "service", "flag", "srcbytes", "dstbytes",
    "land", "wrongfragment", "urgent", "hot", "numfailedlogins", "loggedin",
    "numcompromised", "rootshell", "suattempted", "numroot",
    "numfilecreations", "numshells", "numaccessfiles", "numoutboundcmds",
    "ishostlogin", "isguestlogin", "count", "srvcount", "serrorrate",
    "srvserrorrate", "rerrorrate", "srvrerrorrate", "samesrvrate",
    "diffsrvrate", "srvdiffhostrate", "dsthostcount", "dsthostsrvcount",
    "dsthostsamesrvrate", "dsthostdiffsrvrate", "dsthostsamesrcportrate",
    "dsthostsrvdiffhostrate", "dsthostserrorrate", "dsthostsrvserrorrate",
    "dsthostrerrorrate", "dsthostsrvrerrorrate",
)

ENGINEERED_FEATURES = (
    "total_bytes", "bytes_per_second", "src_dst_byte_ratio",
    "src_byte_fraction", "dst_byte_fraction", "byte_asymmetry",
    "service_connection_ratio", "host_service_ratio",
    "different_service_connections", "different_host_service_connections",
    "short_term_scan_pressure", "host_scan_pressure",
    "same_source_port_pressure", "short_term_serror_score",
    "short_term_rerror_score", "short_term_error_score", "host_serror_score",
    "host_rerror_score", "host_error_score", "same_service_rate_gap",
    "different_service_rate_gap", "serror_rate_gap", "rerror_rate_gap",
    "authentication_risk_score", "has_failed_login",
    "failed_login_and_logged_in", "suspicious_admin_activity",
    "privileged_activity_score", "content_risk_score",
    "file_and_shell_activity", "root_compromise_ratio",
)

COMBINED_FEATURES = ORIGINAL_FEATURES + ENGINEERED_FEATURES

df = pd.read_csv(DATA_PATH)
missing_columns = sorted(set(COMBINED_FEATURES) - set(df.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

X_all = df.loc[:, list(COMBINED_FEATURES)].copy()
y_all = df["binary_target"].astype(str).map({"Normal": 0, "Attack": 1})
if y_all.isna().any():
    raise ValueError("binary_target must contain only Normal and Attack")
y_all = y_all.astype(int)
attack_family_all = df["attack_category"].astype(str)

all_positions = np.arange(len(df))
train_positions, test_positions = train_test_split(
    all_positions,
    test_size=0.20,
    stratify=y_all,
    random_state=RANDOM_STATE,
)

X_train = X_all.iloc[train_positions].copy()
y_train = y_all.iloc[train_positions].to_numpy()
X_test = X_all.iloc[test_positions].copy()
y_test = y_all.iloc[test_positions].to_numpy()
attack_family_test = attack_family_all.iloc[test_positions].to_numpy()

split_summary = pd.DataFrame(
    {
        "rows": [len(X_train), len(X_test)],
        "normal": [int((y_train == 0).sum()), int((y_test == 0).sum())],
        "attack": [int((y_train == 1).sum()), int((y_test == 1).sum())],
    },
    index=["training_pool", "external_test"],
)
display(split_summary)

,rows,normal,attack
training_pool,100778,53874,46904
external_test,25195,13469,11726


## 3. Split only the training pool

- **Model fit:** fits XGBoost and Isolation Forest.
- **Normal reference:** converts Isolation Forest scores to empirical normal percentiles.
- **Routing validation:** learns the low/high/extreme thresholds.
- **External test:** remains untouched until final evaluation.

In [3]:
(
    X_fit,
    X_reference,
    X_validation,
    y_fit,
    y_reference,
    y_validation,
) = split_training_for_hybrid(
    X_train,
    y_train,
    reference_fraction=REFERENCE_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    random_state=RANDOM_STATE,
)

internal_split_summary = pd.DataFrame(
    {
        "rows": [len(X_fit), len(X_reference), len(X_validation), len(X_test)],
        "normal": [
            int((y_fit == 0).sum()),
            int((y_reference == 0).sum()),
            int((y_validation == 0).sum()),
            int((y_test == 0).sum()),
        ],
        "attack": [
            int((y_fit == 1).sum()),
            int((y_reference == 1).sum()),
            int((y_validation == 1).sum()),
            int((y_test == 1).sum()),
        ],
        "purpose": [
            "fit both branches",
            "normal novelty percentile",
            "learn routing thresholds",
            "final evaluation only",
        ],
    },
    index=["fit", "reference", "validation", "external_test"],
)
display(internal_split_summary)

,rows,normal,attack,purpose
fit,70544,37711,32833,fit both branches
reference,15117,8082,7035,normal novelty percentile
validation,15117,8081,7036,learn routing thresholds
external_test,25195,13469,11726,final evaluation only


## 4. Load the selected Isolation Forest features

This cell reads the exact selected-feature artifact from the latest finished `combined_correlationfiltering_IsolationForest` run. It does **not** fit Correlation Filtering again. This keeps the Stage‑1 Isolation Forest input consistent with the feature-selection experiment.

In [4]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
source_runs = mlflow.search_runs(
    [experiment.experiment_id],
    filter_string=(
        f"tags.mlflow.runName = '{SOURCE_RUN_NAME}' "
        "and attributes.status = 'FINISHED'"
    ),
    order_by=["start_time DESC"],
    max_results=1,
)

if source_runs.empty:
    raise RuntimeError(
        f"MLflow run '{SOURCE_RUN_NAME}' was not found. "
        "Execute feature_selection_Anamoly_Detection.ipynb first."
    )

feature_source_run_id = str(source_runs.iloc[0]["run_id"])
feature_source_uri = f"runs:/{feature_source_run_id}/{SOURCE_ARTIFACT}"
feature_payload = mlflow.artifacts.load_dict(feature_source_uri)

IF_SELECTED_FEATURES = tuple(feature_payload["selected_features_or_components"])
if not IF_SELECTED_FEATURES:
    raise ValueError("The selected-feature artifact is empty")
if len(IF_SELECTED_FEATURES) != len(set(IF_SELECTED_FEATURES)):
    raise ValueError("The selected-feature artifact contains duplicates")

missing_selected = sorted(set(IF_SELECTED_FEATURES) - set(COMBINED_FEATURES))
if missing_selected:
    raise ValueError(f"Selected features are absent from the combined data: {missing_selected}")

selected_feature_table = pd.DataFrame(
    {
        "feature": IF_SELECTED_FEATURES,
        "feature_type": [
            "Engineered" if feature in ENGINEERED_FEATURES else "Original"
            for feature in IF_SELECTED_FEATURES
        ],
    }
)

print("Source run:", feature_source_run_id)
print(f"Isolation Forest raw inputs: {len(IF_SELECTED_FEATURES)} of {len(COMBINED_FEATURES)}")
display(selected_feature_table)

Source run: fcf0eb3c196e4b56905661e26fc01c5b
Isolation Forest raw inputs: 56 of 72


,feature,feature_type
0,duration,Original
1,protocoltype,Original
2,service,Original
3,srcbytes,Original
4,dstbytes,Original
5,land,Original
6,wrongfragment,Original
7,urgent,Original
8,hot,Original
9,numfailedlogins,Original


## 5. Fit calibrated XGBoost on original features

The XGBoost hyperparameters and preprocessing match the binary original-feature experiment. `CalibratedClassifierCV` performs 5-fold sigmoid calibration inside the model-fit partition, so routing receives a calibrated `P(Attack)` without using validation or test labels during model fitting.

In [5]:
xgb_categorical = (
    X_fit.loc[:, list(ORIGINAL_FEATURES)]
    .select_dtypes(include=["object", "category", "string"])
    .columns.tolist()
)
xgb_numeric = [feature for feature in ORIGINAL_FEATURES if feature not in xgb_categorical]

xgb_preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            xgb_numeric,
        ),
        (
            "categorical",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            xgb_categorical,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

XGB_PARAMS = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

xgb_pipeline = Pipeline(
    [
        ("preprocessor", xgb_preprocessor),
        ("model", XGBClassifier(**XGB_PARAMS)),
    ]
)

calibrated_xgboost = CalibratedClassifierCV(
    estimator=xgb_pipeline,
    method="sigmoid",
    cv=CALIBRATION_FOLDS,
    ensemble=False,
    n_jobs=-1,
)

xgb_started = time.perf_counter()
calibrated_xgboost.fit(X_fit.loc[:, list(ORIGINAL_FEATURES)], y_fit)
xgb_fit_seconds = time.perf_counter() - xgb_started

validation_xgb_probability = calibrated_xgboost.predict_proba(
    X_validation.loc[:, list(ORIGINAL_FEATURES)]
)[:, 1]

print(f"Calibrated XGBoost fit time: {xgb_fit_seconds:.2f} seconds")
print(f"Validation Brier score: {brier_score_loss(y_validation, validation_xgb_probability):.6f}")

Calibrated XGBoost fit time: 23.39 seconds
Validation Brier score: 0.001002


## 6. Fit Isolation Forest on trusted Normal traffic

Only Normal rows from the model-fit partition train Isolation Forest. Its raw anomaly scores are converted to percentiles using separate Normal rows from the reference partition. A percentile near 1 means the observation is more anomalous than almost all trusted reference traffic.

In [6]:
if_categorical = (
    X_fit.loc[:, list(IF_SELECTED_FEATURES)]
    .select_dtypes(include=["object", "category", "string"])
    .columns.tolist()
)
if_numeric = [feature for feature in IF_SELECTED_FEATURES if feature not in if_categorical]

if_preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            if_numeric,
        ),
        (
            "categorical",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            if_categorical,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

IFOREST_PARAMS = {
    "n_estimators": 300,
    "max_samples": 1.0,
    "contamination": "auto",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

normal_iforest = Pipeline(
    [
        ("preprocessor", if_preprocessor),
        ("detector", IsolationForest(**IFOREST_PARAMS)),
    ]
)

normal_fit = X_fit.loc[y_fit == 0, list(IF_SELECTED_FEATURES)].copy()
if len(normal_fit) > NORMAL_FIT_LIMIT:
    normal_fit = normal_fit.sample(n=NORMAL_FIT_LIMIT, random_state=RANDOM_STATE)

iforest_started = time.perf_counter()
normal_iforest.fit(normal_fit)
iforest_fit_seconds = time.perf_counter() - iforest_started

normal_reference = X_reference.loc[y_reference == 0, list(IF_SELECTED_FEATURES)]
reference_raw_novelty = raw_iforest_anomaly_score(normal_iforest, normal_reference)
novelty_calibrator = NormalNoveltyPercentile().fit(reference_raw_novelty)

validation_raw_novelty = raw_iforest_anomaly_score(
    normal_iforest,
    X_validation.loc[:, list(IF_SELECTED_FEATURES)],
)
validation_novelty_percentile = novelty_calibrator.transform(validation_raw_novelty)

encoded_dimensions = normal_iforest.named_steps["preprocessor"].transform(normal_fit.head(1)).shape[1]
print(f"Isolation Forest fit time: {iforest_fit_seconds:.2f} seconds")
print(f"Trusted Normal fit rows: {len(normal_fit)}")
print(f"Normal reference rows: {len(normal_reference)}")
print(f"Selected raw features: {len(IF_SELECTED_FEATURES)}; encoded dimensions: {encoded_dimensions}")

Isolation Forest fit time: 0.72 seconds
Trusted Normal fit rows: 6000
Normal reference rows: 8082
Selected raw features: 56; encoded dimensions: 79


## 7. Learn routing thresholds on validation data

Thresholds are learned from error budgets:

- `xgb_high`: maximize attack recall while keeping automatic Stage‑2 false positives within the XGBoost FPR budget.
- `xgb_low`: largest auto-Normal boundary satisfying both attack escape constraints.
- `novelty_high`: high-novelty boundary within its Normal FPR budget.
- `novelty_extreme`: stricter cutoff for novel-anomaly candidates.

In [7]:
routing_thresholds, threshold_diagnostics = learn_routing_thresholds(
    y_validation,
    validation_xgb_probability,
    validation_novelty_percentile,
    **THRESHOLD_BUDGETS,
)

threshold_table = pd.DataFrame(
    {
        "threshold": asdict(routing_thresholds),
        "meaning": {
            "xgb_low": "at or below: supervised low-risk region",
            "xgb_high": "at or above: automatically route to Stage 2",
            "novelty_high": "at or above: high Isolation Forest novelty",
            "novelty_extreme": "at or above: extreme novelty",
        },
    }
)

diagnostic_table = pd.DataFrame.from_dict(
    asdict(threshold_diagnostics), orient="index", columns=["validation_value"]
)

display(threshold_table)
display(diagnostic_table)

,threshold,meaning
xgb_low,0.007929,at or below: supervised low-risk region
xgb_high,0.110088,at or above: automatically route to Stage 2
novelty_high,0.991092,at or above: high Isolation Forest novelty
novelty_extreme,0.999876,at or above: extreme novelty


,validation_value
xgb_high_fpr,0.000990
xgb_high_recall,0.998721
xgb_high_precision,0.998863
xgb_low_attack_escape_rate,0.000995
xgb_low_region_attack_rate,0.000867
xgb_low_coverage,0.534035
novelty_high_fpr,0.009776
novelty_extreme_fpr,0.000495


## 8. Build the Stage‑1 router and evaluate once on the external test set

Routing policy:

| XGBoost region | Isolation Forest region | Route |
|---|---|---|
| High | Any | `ROUTE_STAGE_2` |
| Low | Below high novelty | `RETURN_NORMAL` |
| Low | Extreme novelty | `NOVEL_ANOMALY_CANDIDATE` |
| Remaining combinations | Any | `REVIEW_DISAGREEMENT` |

The novelty branch can flag unfamiliar behavior, but this dataset does not contain explicit unknown-class ground truth. Therefore, `NOVEL_ANOMALY_CANDIDATE` means suspicious and out-of-distribution—not a confirmed unknown attack.

In [8]:
hybrid_router = HybridRouter(
    original_features=tuple(ORIGINAL_FEATURES),
    combined_features=tuple(IF_SELECTED_FEATURES),
    calibrated_xgboost=calibrated_xgboost,
    normal_iforest=normal_iforest,
    novelty_calibrator=novelty_calibrator,
    thresholds=routing_thresholds,
    model_version=MODEL_VERSION,
    threshold_version=THRESHOLD_VERSION,
)

test_routes = hybrid_router.score(X_test)
test_routes["binary_target"] = y_test
test_routes["true_label"] = np.where(y_test == 1, "Attack", "Normal")
test_routes["attack_category"] = attack_family_test

def binary_metrics(y_true, prediction, score, prefix):
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        f"{prefix}_accuracy": accuracy_score(y_true, prediction),
        f"{prefix}_balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        f"{prefix}_precision": precision_score(y_true, prediction, zero_division=0),
        f"{prefix}_recall": recall_score(y_true, prediction, zero_division=0),
        f"{prefix}_f1": f1_score(y_true, prediction, zero_division=0),
        f"{prefix}_roc_auc": roc_auc_score(y_true, score),
        f"{prefix}_average_precision": average_precision_score(y_true, score),
        f"{prefix}_false_positive_rate": fp / max(fp + tn, 1),
    }

xgb_test_probability = test_routes["xgb_attack_probability"].to_numpy()
novelty_test = test_routes["normal_novelty_percentile"].to_numpy()
route_test = test_routes["route"].to_numpy()

xgb_prediction = (xgb_test_probability >= 0.50).astype(int)
iforest_prediction = (novelty_test >= routing_thresholds.novelty_high).astype(int)

automatic_mask = route_test != "REVIEW_DISAGREEMENT"
automatic_prediction = np.isin(
    route_test[automatic_mask],
    ["ROUTE_STAGE_2", "NOVEL_ANOMALY_CANDIDATE"],
).astype(int)

# Conservative monitoring view: reviews are counted as alerts, never auto-Normal.
conservative_prediction = (route_test != "RETURN_NORMAL").astype(int)

metrics = {
    **binary_metrics(y_test, xgb_prediction, xgb_test_probability, "xgboost"),
    **binary_metrics(y_test, iforest_prediction, novelty_test, "isolation_forest"),
    **binary_metrics(y_test, conservative_prediction, xgb_test_probability, "hybrid_conservative"),
    "xgboost_brier": brier_score_loss(y_test, xgb_test_probability),
    "automatic_coverage": float(automatic_mask.mean()),
    "automatic_accuracy": accuracy_score(y_test[automatic_mask], automatic_prediction),
    "review_rate": float((route_test == "REVIEW_DISAGREEMENT").mean()),
    "attack_stage2_route_rate": float((route_test[y_test == 1] == "ROUTE_STAGE_2").mean()),
    "attack_novel_candidate_rate": float((route_test[y_test == 1] == "NOVEL_ANOMALY_CANDIDATE").mean()),
    "attack_review_rate": float((route_test[y_test == 1] == "REVIEW_DISAGREEMENT").mean()),
    "normal_return_rate": float((route_test[y_test == 0] == "RETURN_NORMAL").mean()),
    "normal_novel_candidate_rate": float((route_test[y_test == 0] == "NOVEL_ANOMALY_CANDIDATE").mean()),
    "xgboost_fit_seconds": xgb_fit_seconds,
    "isolation_forest_fit_seconds": iforest_fit_seconds,
}

route_distribution = pd.crosstab(
    test_routes["true_label"],
    test_routes["route"],
    normalize="index",
)
attack_family_routes = pd.crosstab(
    test_routes["attack_category"],
    test_routes["route"],
)

STAGE1_OUTPUT_COLUMNS = [
    "xgb_attack_probability",
    "xgb_risk_band",
    "normal_novelty_percentile",
    "novelty_band",
    "route",
    "route_reason",
    "xgb_low_threshold",
    "xgb_high_threshold",
    "novelty_high_threshold",
    "novelty_extreme_threshold",
    "model_version",
    "threshold_version",
]

novel_examples = test_routes.index[
    test_routes["route"] == "NOVEL_ANOMALY_CANDIDATE"
].tolist()
example_index = novel_examples[0] if novel_examples else test_routes.index[0]
stage1_output_example = json.loads(
    test_routes.loc[example_index, STAGE1_OUTPUT_COLUMNS].to_json()
)

display(pd.DataFrame.from_dict(metrics, orient="index", columns=["test_value"]))
display(route_distribution)
display(attack_family_routes)
display(test_routes.loc[:, STAGE1_OUTPUT_COLUMNS].head(10))
print("Stage-1 JSON output example:")
print(json.dumps(stage1_output_example, indent=2))

,test_value
xgboost_accuracy,0.999167
xgboost_balanced_accuracy,0.999165
xgboost_precision,0.999062
xgboost_recall,0.999147
xgboost_f1,0.999105
xgboost_roc_auc,0.999992
xgboost_average_precision,0.999991
xgboost_false_positive_rate,0.000817
isolation_forest_accuracy,0.930026
isolation_forest_balanced_accuracy,0.925482


route,NOVEL_ANOMALY_CANDIDATE,RETURN_NORMAL,REVIEW_DISAGREEMENT,ROUTE_STAGE_2
true_label,,,,
Attack,0.000000,0.000341,0.000341,0.999318
Normal,0.000148,0.990348,0.008538,0.000965


route,NOVEL_ANOMALY_CANDIDATE,RETURN_NORMAL,REVIEW_DISAGREEMENT,ROUTE_STAGE_2
attack_category,,,,
DoS,0,0,0,9105
Normal,2,13339,115,13
Probe,0,2,2,2385
R2L,0,2,2,222
U2R,0,0,0,6


,xgb_attack_probability,xgb_risk_band,normal_novelty_percentile,novelty_band,route,route_reason,xgb_low_threshold,xgb_high_threshold,novelty_high_threshold,novelty_extreme_threshold,model_version,threshold_version
0,0.999522,High,0.999629,High,ROUTE_STAGE_2,High supervised attack risk,0.007929,0.110088,0.991092,0.999876,stage1-router-v1,routing-thresholds-v1
1,0.001061,Low,0.066436,Low,RETURN_NORMAL,Low supervised risk and low deviation from normal,0.007929,0.110088,0.991092,0.999876,stage1-router-v1,routing-thresholds-v1
2,0.999522,High,0.999505,High,ROUTE_STAGE_2,High supervised attack risk,0.007929,0.110088,0.991092,0.999876,stage1-router-v1,routing-thresholds-v1
3,0.999522,High,0.999876,ExtremelyHigh,ROUTE_STAGE_2,High supervised attack risk,0.007929,0.110088,0.991092,0.999876,stage1-router-v1,routing-thresholds-v1
4,0.001060,Low,0.094643,Low,RETURN_NORMAL,Low supervised risk and low deviation from normal,0.007929,0.110088,0.991092,0.999876,stage1-router-v1,routing-thresholds-v1
5,0.001060,Low,0.563034,Low,RETURN_NORMAL,Low supervised risk and low deviation from normal,0.007929,0.110088,0.991092,0.999876,stage1-router-v1,routing-thresholds-v1
6,0.001061,Low,0.619325,Low,RETURN_NORMAL,Low supervised risk and low deviation from normal,0.007929,0.110088,0.991092,0.999876,stage1-router-v1,routing-thresholds-v1
7,0.001060,Low,0.212297,Low,RETURN_NORMAL,Low supervised risk and low deviation from normal,0.007929,0.110088,0.991092,0.999876,stage1-router-v1,routing-thresholds-v1
8,0.999522,High,0.999505,High,ROUTE_STAGE_2,High supervised attack risk,0.007929,0.110088,0.991092,0.999876,stage1-router-v1,routing-thresholds-v1
9,0.999522,High,0.999505,High,ROUTE_STAGE_2,High supervised attack risk,0.007929,0.110088,0.991092,0.999876,stage1-router-v1,routing-thresholds-v1


Stage-1 JSON output example:
{
  "xgb_attack_probability": 0.0010882054,
  "xgb_risk_band": "Low",
  "normal_novelty_percentile": 1.0,
  "novelty_band": "ExtremelyHigh",
  "route": "NOVEL_ANOMALY_CANDIDATE",
  "route_reason": "Low supervised risk but extreme deviation from normal",
  "xgb_low_threshold": 0.0079290098,
  "xgb_high_threshold": 0.1100879543,
  "novelty_high_threshold": 0.9910924162,
  "novelty_extreme_threshold": 0.9998762836,
  "model_version": "stage1-router-v1",
  "threshold_version": "routing-thresholds-v1"
}


## 9. Visual diagnostics

In [9]:
fig_calibration, ax = plt.subplots(figsize=(6, 5))
CalibrationDisplay.from_predictions(
    y_test,
    xgb_test_probability,
    n_bins=10,
    strategy="quantile",
    name="Calibrated XGBoost",
    ax=ax,
)
ax.set_title("Stage-1 XGBoost probability calibration")
fig_calibration.tight_layout()
display(fig_calibration)

sample_size = min(6000, len(test_routes))
sample_positions = np.random.default_rng(RANDOM_STATE).choice(
    len(test_routes), size=sample_size, replace=False
)
fig_risk_plane, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    xgb_test_probability[sample_positions],
    novelty_test[sample_positions],
    c=y_test[sample_positions],
    cmap="coolwarm",
    s=9,
    alpha=0.5,
)
ax.axvline(routing_thresholds.xgb_low, color="grey", linestyle="--", label="XGB low")
ax.axvline(routing_thresholds.xgb_high, color="red", linestyle="--", label="XGB high")
ax.axhline(routing_thresholds.novelty_high, color="orange", linestyle=":", label="Novelty high")
ax.axhline(routing_thresholds.novelty_extreme, color="purple", linestyle="--", label="Novelty extreme")
ax.set_xlabel("Calibrated XGBoost P(Attack)")
ax.set_ylabel("Normal-reference novelty percentile")
ax.set_title("Hybrid routing score plane")
ax.legend(loc="lower right")
fig_risk_plane.colorbar(scatter, ax=ax, label="True binary label")
fig_risk_plane.tight_layout()
display(fig_risk_plane)

fig_routes, ax = plt.subplots(figsize=(10, 4))
route_distribution.plot(kind="bar", stacked=True, ax=ax)
ax.set_ylabel("Fraction within true class")
ax.set_title("Stage-1 routes by true binary label")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
fig_routes.tight_layout()
display(fig_routes)

fig_confusion, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    conservative_prediction,
    display_labels=["Normal", "Attack/Review"],
    cmap="Blues",
    colorbar=False,
    ax=ax,
)
ax.set_title("Conservative Stage-1 monitoring view")
fig_confusion.tight_layout()
display(fig_confusion)

<Figure size 600x500 with 1 Axes>

<Figure size 800x600 with 2 Axes>

<Figure size 1000x400 with 1 Axes>

<Figure size 500x400 with 1 Axes>

## 10. Log the complete Stage‑1 experiment to MLflow

In [10]:
fit_tracking = X_fit.copy()
fit_tracking["binary_target"] = np.where(y_fit == 1, "Attack", "Normal")
reference_tracking = X_reference.copy()
reference_tracking["binary_target"] = np.where(y_reference == 1, "Attack", "Normal")
validation_tracking = X_validation.copy()
validation_tracking["binary_target"] = np.where(y_validation == 1, "Attack", "Normal")
test_tracking = X_test.copy()
test_tracking["binary_target"] = np.where(y_test == 1, "Attack", "Normal")

datasets = {
    "model_fit": mlflow.data.from_pandas(
        fit_tracking,
        source=str(DATA_PATH.resolve()),
        targets="binary_target",
        name="hybrid_model_fit",
    ),
    "normal_reference": mlflow.data.from_pandas(
        reference_tracking,
        source=str(DATA_PATH.resolve()),
        targets="binary_target",
        name="hybrid_normal_reference",
    ),
    "routing_validation": mlflow.data.from_pandas(
        validation_tracking,
        source=str(DATA_PATH.resolve()),
        targets="binary_target",
        name="hybrid_routing_validation",
    ),
    "external_test": mlflow.data.from_pandas(
        test_tracking,
        source=str(DATA_PATH.resolve()),
        targets="binary_target",
        name="hybrid_external_test",
    ),
}

with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.set_tags(
        {
            "task": "hybrid_stage1_binary_router",
            "supervised_branch": "original_features_calibrated_xgboost",
            "novelty_branch": "upstream_correlation_selected_combined_isolation_forest",
            "threshold_source": "routing_validation_error_budgets",
            "feature_selection_source_run_id": feature_source_run_id,
            "external_test_used_for_thresholds": "false",
            "model_version": MODEL_VERSION,
            "threshold_version": THRESHOLD_VERSION,
        }
    )

    for context, dataset in datasets.items():
        mlflow.log_input(dataset, context=context)

    mlflow.log_params(
        {
            **{f"xgb_{key}": value for key, value in XGB_PARAMS.items()},
            **{f"iforest_{key}": value for key, value in IFOREST_PARAMS.items()},
            **THRESHOLD_BUDGETS,
            "model_version": MODEL_VERSION,
            "threshold_version": THRESHOLD_VERSION,
            "calibration_class": "CalibratedClassifierCV",
            "calibration_method": "sigmoid",
            "calibration_folds": CALIBRATION_FOLDS,
            "original_feature_count": len(ORIGINAL_FEATURES),
            "combined_feature_count": len(COMBINED_FEATURES),
            "iforest_selected_raw_feature_count": len(IF_SELECTED_FEATURES),
            "iforest_encoded_dimensions": encoded_dimensions,
            "normal_fit_limit": NORMAL_FIT_LIMIT,
            "normal_fit_rows": len(normal_fit),
            "normal_reference_rows": len(normal_reference),
            "fit_rows": len(X_fit),
            "validation_rows": len(X_validation),
            "external_test_rows": len(X_test),
            "random_state": RANDOM_STATE,
            "feature_selection_source_run_id": feature_source_run_id,
        }
    )

    mlflow.log_metrics({key: float(value) for key, value in metrics.items()})
    mlflow.log_metrics(
        {f"threshold_{key}": float(value) for key, value in asdict(routing_thresholds).items()}
    )
    mlflow.log_metrics(
        {f"validation_{key}": float(value) for key, value in asdict(threshold_diagnostics).items()}
    )

    mlflow.log_dict(
        {
            "selected_features": list(IF_SELECTED_FEATURES),
            "selected_engineered_features": [
                feature for feature in IF_SELECTED_FEATURES if feature in ENGINEERED_FEATURES
            ],
            "source_run_name": SOURCE_RUN_NAME,
            "source_run_id": feature_source_run_id,
            "source_artifact_uri": feature_source_uri,
            "source_selection_details": feature_payload.get("selection_details", {}),
        },
        "feature_selection/isolation_forest_selected_features.json",
    )
    mlflow.log_dict(
        {
            "thresholds": asdict(routing_thresholds),
            "validation_diagnostics": asdict(threshold_diagnostics),
            "budgets": THRESHOLD_BUDGETS,
            "policy": {
                "high_xgboost": "ROUTE_STAGE_2",
                "low_xgboost_and_low_novelty": "RETURN_NORMAL",
                "low_xgboost_and_extreme_novelty": "NOVEL_ANOMALY_CANDIDATE",
                "otherwise": "REVIEW_DISAGREEMENT",
            },
        },
        "routing/thresholds_and_policy.json",
    )
    mlflow.log_dict(
        {
            "required_input_columns": list(COMBINED_FEATURES),
            "xgboost_input_columns": list(ORIGINAL_FEATURES),
            "isolation_forest_input_columns": list(IF_SELECTED_FEATURES),
            "output_columns": STAGE1_OUTPUT_COLUMNS,
            "output_format": "one JSON object per input record",
            "model_version": MODEL_VERSION,
            "threshold_version": THRESHOLD_VERSION,
            "route_values": [
                "RETURN_NORMAL",
                "ROUTE_STAGE_2",
                "NOVEL_ANOMALY_CANDIDATE",
                "REVIEW_DISAGREEMENT",
            ],
        },
        "metadata/input_output_contract.json",
    )
    mlflow.log_dict(
        {
            "python": platform.python_version(),
            "sklearn": sklearn.__version__,
            "xgboost": xgboost.__version__,
            "mlflow": mlflow.__version__,
            "split_design": "external 20% test; training pool split 70/15/15 fit/reference/validation",
            "unknown_class_ground_truth_available": False,
        },
        "metadata/run_metadata.json",
    )

    mlflow.log_table(route_distribution.reset_index(), "routing/test_route_distribution.json")
    mlflow.log_table(attack_family_routes.reset_index(), "routing/test_routes_by_attack_family.json")
    mlflow.log_table(test_routes.head(1000), "examples/test_router_output_sample.json")
    mlflow.log_dict(stage1_output_example, "examples/stage1_output_example.json")

    mlflow.log_figure(fig_calibration, "plots/xgboost_calibration.png")
    mlflow.log_figure(fig_risk_plane, "plots/hybrid_risk_plane.png")
    mlflow.log_figure(fig_routes, "plots/route_distribution.png")
    mlflow.log_figure(fig_confusion, "plots/conservative_confusion_matrix.png")

    xgb_example = X_test.loc[:, list(ORIGINAL_FEATURES)].head(5)
    xgb_signature = infer_signature(
        xgb_example,
        calibrated_xgboost.predict_proba(xgb_example),
    )
    mlflow.sklearn.log_model(
        calibrated_xgboost,
        name="calibrated_xgboost",
        signature=xgb_signature,
        input_example=xgb_example,
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
    )

    iforest_example = X_test.loc[:, list(IF_SELECTED_FEATURES)].head(5)
    iforest_signature = infer_signature(
        iforest_example,
        raw_iforest_anomaly_score(normal_iforest, iforest_example),
    )
    mlflow.sklearn.log_model(
        normal_iforest,
        name="normal_only_isolation_forest",
        signature=iforest_signature,
        input_example=iforest_example,
        serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
    )

    with tempfile.TemporaryDirectory() as temporary_directory:
        router_path = Path(temporary_directory) / "hybrid_stage1_router.joblib"
        joblib.dump(hybrid_router, router_path)
        mlflow.log_artifact(str(router_path), artifact_path="router")

    hybrid_run_id = run.info.run_id

for figure in [fig_calibration, fig_risk_plane, fig_routes, fig_confusion]:
    plt.close(figure)

print("Finished MLflow run:", hybrid_run_id)

D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


2026/08/12 08:35:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/08/12 08:35:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can

Finished MLflow run: 009e649215464979b79cc2bd206e8119


## 11. Measured Stage‑1 conclusion

In [11]:
display(
    Markdown(
        f"""
### Selected thresholds

- XGBoost low boundary: **{routing_thresholds.xgb_low:.6f}**
- XGBoost high boundary: **{routing_thresholds.xgb_high:.6f}**
- Isolation Forest high-novelty percentile: **{routing_thresholds.novelty_high:.6f}**
- Isolation Forest extreme-novelty percentile: **{routing_thresholds.novelty_extreme:.6f}**

These values were selected from the routing-validation partition under the declared error budgets. They were not tuned on the external test set.

### External-test behavior

The calibrated original-feature XGBoost branch achieves F1 **{metrics['xgboost_f1']:.4f}**, recall **{metrics['xgboost_recall']:.4f}**, and FPR **{metrics['xgboost_false_positive_rate']:.4f}**. The correlation-selected Isolation Forest branch achieves F1 **{metrics['isolation_forest_f1']:.4f}**, recall **{metrics['isolation_forest_recall']:.4f}**, and FPR **{metrics['isolation_forest_false_positive_rate']:.4f}**.

The router automatically resolves **{metrics['automatic_coverage']:.2%}** of traffic with **{metrics['automatic_accuracy']:.4f}** accuracy on those automatically resolved rows. It routes **{metrics['attack_stage2_route_rate']:.2%}** of known attacks directly to Stage 2, sends **{metrics['attack_review_rate']:.2%}** of known attacks to review, and automatically returns **{metrics['normal_return_rate']:.2%}** of Normal traffic.

### Decision

Use these thresholds as the measured baseline for Stage 1, then re-estimate them when the acceptable false-positive rate, missed-attack rate, or analyst review capacity changes. Treat `NOVEL_ANOMALY_CANDIDATE` as an investigation queue until leave-one-attack-family-out or temporal out-of-distribution validation demonstrates unknown-attack detection capability.

**MLflow run:** `{hybrid_run_id}`
"""
    )
)


### Selected thresholds

- XGBoost low boundary: **0.007929**
- XGBoost high boundary: **0.110088**
- Isolation Forest high-novelty percentile: **0.991092**
- Isolation Forest extreme-novelty percentile: **0.999876**

These values were selected from the routing-validation partition under the declared error budgets. They were not tuned on the external test set.

### External-test behavior

The calibrated original-feature XGBoost branch achieves F1 **0.9991**, recall **0.9991**, and FPR **0.0008**. The correlation-selected Isolation Forest branch achieves F1 **0.9196**, recall **0.8598**, and FPR **0.0088**.

The router automatically resolves **99.53%** of traffic with **0.9992** accuracy on those automatically resolved rows. It routes **99.93%** of known attacks directly to Stage 2, sends **0.03%** of known attacks to review, and automatically returns **99.03%** of Normal traffic.

### Decision

Use these thresholds as the measured baseline for Stage 1, then re-estimate them when the acceptable false-positive rate, missed-attack rate, or analyst review capacity changes. Treat `NOVEL_ANOMALY_CANDIDATE` as an investigation queue until leave-one-attack-family-out or temporal out-of-distribution validation demonstrates unknown-attack detection capability.

**MLflow run:** `009e649215464979b79cc2bd206e8119`


# Hybrid Stage 2: Attack Classification + Open-Set Novelty

Stage 2 consumes the scores and route produced by Stage 1 and combines three attack-only models:

1. **Model 1 — calibrated Random Forest:** predicts `DoS`, `Probe`, `R2L`, or `U2R` using the 41 original features. Its Random Forest parameters come from the winning `Original_attackclass_randomforest` experiment in `Experiment.ipynb`.
2. **Model 2 — global attack novelty:** One-Class SVM trained on all known attack families. This is the winner from the `Original_global_attack_{Algorithm}` experiments in `Anamoly Detection.ipynb`.
3. **Model 3 — predicted-category novelty:** DoS, R2L, and U2R use One-Class SVM; Probe uses Isolation Forest, matching the winning `Original_{Attack_category}_{Algorithm}` runs.

The implementation follows `stage2_router.py`. Stage‑2 route thresholds are learned only from the validation partition. The external test partition is used once for final evaluation.

## 12. Stage‑2 imports, provenance, and model configuration

The category classifier uses every attack row in the model-fit partition. Global attack novelty uses a balanced maximum of 6,000 known attacks plus a separate known-attack reference partition. Category novelty models use their own family-only fit/reference splits.

In [12]:
from dataclasses import asdict
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import OneClassSVM

from stage2_router import (
    AttackCategoryNoveltyBank,
    CompleteStage2Router,
    DataQualityConfig,
    Stage2OpenSetRouter,
    Stage2PolicyConfig,
    evaluate_stage2,
    stage2_classification_report,
)

STAGE2_RUN_NAME = "Hybrid_Stage2_RandomForest_OpenSetNovelty"
STAGE2_MODEL_VERSION = "stage2-router-v1"
STAGE2_CATEGORY_NOVELTY_VERSION = "stage2-category-novelty-v1"

# Exact winning attack-only Random Forest parameters from Experiment.ipynb.
ATTACK_RF_PARAMS = {
    "n_estimators": 200,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

# Winning Model 2 configuration from Anamoly Detection.ipynb.
GLOBAL_ATTACK_NOVELTY_PARAMS = {
    "kernel": "rbf",
    "nu": 0.05,
    "gamma": "scale",
}

# Winning Model 3 algorithm for each predicted family.
CATEGORY_NOVELTY_ALGORITHMS = {
    "DoS": "OneClassSVM",
    "Probe": "IsolationForest",
    "R2L": "OneClassSVM",
    "U2R": "OneClassSVM",
}

PROVENANCE_RUN_NAMES = {
    "supervised_attack_classifier": "Original_attackclass_randomforest",
    "global_attack_novelty": "Original_global_attack_oneclasssvm",
    "category_DoS": "Original_Dos_oneclasssvm",
    "category_Probe": "Original_Probe_isolationforest",
    "category_R2L": "Original_R2L_oneclasssvm",
    "category_U2R": "Original_U2R_oneclasssvm",
}

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
provenance_run_ids = {}
for role, source_run_name in PROVENANCE_RUN_NAMES.items():
    source = mlflow.search_runs(
        [experiment.experiment_id],
        filter_string=(
            f"tags.mlflow.runName = '{source_run_name}' "
            "and attributes.status = 'FINISHED'"
        ),
        order_by=["start_time DESC"],
        max_results=1,
    )
    if source.empty:
        raise RuntimeError(f"Required source run was not found: {source_run_name}")
    provenance_run_ids[role] = str(source.iloc[0]["run_id"])

display(pd.DataFrame({
    "component": provenance_run_ids.keys(),
    "source_run_name": PROVENANCE_RUN_NAMES.values(),
    "source_run_id": provenance_run_ids.values(),
}))


,component,source_run_name,source_run_id
0,supervised_attack_classifier,Original_attackclass_randomforest,2e05768f7e1444b2917118337a8ccf95
1,global_attack_novelty,Original_global_attack_oneclasssvm,1d3ecbadc26a456cba7bafdf793ca70b
2,category_DoS,Original_Dos_oneclasssvm,dcce917c4d0f4071b0e4437eb002da10
3,category_Probe,Original_Probe_isolationforest,c5835846bfeb48108904a8cb03f6a776
4,category_R2L,Original_R2L_oneclasssvm,65b4fc481c54490aa94a8f43ab2a2d30
5,category_U2R,Original_U2R_oneclasssvm,86f2813a1eaa433a9a96bcbea67ce4bc


## 13. Prepare attack-only targets and Stage‑1 evidence

Stage 2 receives the full Stage‑1 output but fits its classifier and novelty components only on attack records. Normal rows remain in validation and test evaluation to verify that Stage‑2 respects Stage‑1 final-Normal decisions and handles disagreement routes correctly.

In [13]:
ATTACK_CATEGORIES = ["DoS", "Probe", "R2L", "U2R"]

family_fit = df.loc[X_fit.index, "attack_category"].astype(str).to_numpy()
family_reference = df.loc[X_reference.index, "attack_category"].astype(str).to_numpy()
family_validation = df.loc[X_validation.index, "attack_category"].astype(str).to_numpy()
family_test_stage2 = df.loc[X_test.index, "attack_category"].astype(str).to_numpy()

y_fit_binary_stage2 = (family_fit != "Normal").astype(int)
y_reference_binary_stage2 = (family_reference != "Normal").astype(int)
y_validation_binary_stage2 = (family_validation != "Normal").astype(int)
y_test_binary_stage2 = (family_test_stage2 != "Normal").astype(int)

def stage1_evidence_for(frame):
    details = hybrid_router.score(frame)
    details.index = frame.index
    return pd.DataFrame(
        {
            "stage1_route": details["route"].astype(str),
            "xgb_attack_probability": details["xgb_attack_probability"].astype(float),
            "normal_novelty_percentile": details["normal_novelty_percentile"].astype(float),
        },
        index=frame.index,
    )

stage1_validation = stage1_evidence_for(X_validation)
stage1_test = stage1_evidence_for(X_test)

stage2_split_summary = pd.DataFrame(
    {
        "rows": [len(X_fit), len(X_reference), len(X_validation), len(X_test)],
        "attack_rows": [
            int(y_fit_binary_stage2.sum()),
            int(y_reference_binary_stage2.sum()),
            int(y_validation_binary_stage2.sum()),
            int(y_test_binary_stage2.sum()),
        ],
        "purpose": [
            "fit classifier/global/category components",
            "global attack novelty reference",
            "fit route-specific Stage-2 policy",
            "final evaluation only",
        ],
    },
    index=["fit", "reference", "validation", "external_test"],
)
display(stage2_split_summary)
display(stage1_validation["stage1_route"].value_counts().rename("validation_rows"))


,rows,attack_rows,purpose
fit,70544,32833,fit classifier/global/category components
reference,15117,7035,global attack novelty reference
validation,15117,7036,fit route-specific Stage-2 policy
external_test,25195,11726,final evaluation only


stage1_route
RETURN_NORMAL              8000
ROUTE_STAGE_2              7035
REVIEW_DISAGREEMENT          78
NOVEL_ANOMALY_CANDIDATE       4
Name: validation_rows, dtype: int64

## 14. Fit Model 1 and Model 2

`CalibratedClassifierCV` applies five-fold sigmoid calibration to the attack-only Random Forest. Model 2 is fitted as the selected global One-Class SVM with StandardScaler preprocessing and no additional correlation filtering, matching its winning experiment.

In [14]:
stage2_policy = Stage2PolicyConfig(
    route_stage2_known_rejection_rate=0.02,
    novel_route_known_rejection_rate=0.10,
    review_route_known_rejection_rate=0.05,
    minimum_route_calibration_samples=40,
)

stage2_quality = DataQualityConfig(
    rate_columns=(
        "serrorrate", "srvserrorrate", "rerrorrate", "srvrerrorrate",
        "samesrvrate", "diffsrvrate", "srvdiffhostrate",
        "dsthostsamesrvrate", "dsthostdiffsrvrate",
        "dsthostsamesrcportrate", "dsthostsrvdiffhostrate",
        "dsthostserrorrate", "dsthostsrvserrorrate",
        "dsthostrerrorrate", "dsthostsrvrerrorrate",
    ),
    nonnegative_columns=(
        "duration", "srcbytes", "dstbytes", "count", "srvcount",
        "dsthostcount", "dsthostsrvcount",
    ),
    binary_columns=("land", "loggedin", "rootshell", "ishostlogin", "isguestlogin"),
    maximum_row_missing_fraction=0.25,
)

stage2_open_router = Stage2OpenSetRouter(
    category_feature_columns=ORIGINAL_FEATURES,
    attack_novelty_feature_columns=ORIGINAL_FEATURES,
    category_model=RandomForestClassifier(**ATTACK_RF_PARAMS),
    attack_novelty_model=OneClassSVM(**GLOBAL_ATTACK_NOVELTY_PARAMS),
    category_calibration_method="sigmoid",
    category_calibration_folds=5,
    category_onehot_min_frequency=None,
    attack_onehot_min_frequency=None,
    attack_apply_feature_filters=False,
    attack_numeric_scaler="standard",
    attack_novelty_fit_limit=6000,
    policy_config=stage2_policy,
    data_quality_config=stage2_quality,
    random_state=RANDOM_STATE,
)

attack_reference_frame = X_reference.loc[
    y_reference_binary_stage2 == 1,
    list(ORIGINAL_FEATURES),
].copy()

component_started = time.perf_counter()
stage2_open_router.fit_components(
    X_fit.loc[:, list(ORIGINAL_FEATURES)],
    y_fit_binary_stage2,
    family_fit,
    X_attack_reference=attack_reference_frame,
)
component_fit_seconds = time.perf_counter() - component_started

print(f"Model 1 + Model 2 fit time: {component_fit_seconds:.2f} seconds")
print("Known attack classes:", stage2_open_router.known_categories_)
display(pd.DataFrame([stage2_open_router.component_fit_metadata_]))


Model 1 + Model 2 fit time: 7.82 seconds
Known attack classes: ['DoS', 'Probe', 'R2L', 'U2R']


,category_training_rows,attack_novelty_training_rows,attack_reference_rows,known_categories
0,32833,6000,7035,"[DoS, Probe, R2L, U2R]"


## 15. Fit Model 3: category novelty bank

Each category detector is trained only on examples from that category. At inference time, only the detector corresponding to the Random Forest’s predicted category is evaluated.

In [15]:
attack_fit_mask = y_fit_binary_stage2 == 1

category_novelty_bank = AttackCategoryNoveltyBank(
    feature_columns=ORIGINAL_FEATURES,
    algorithm_by_category=CATEGORY_NOVELTY_ALGORITHMS,
    threshold_quantile=0.99,
    reference_fraction=0.20,
    fit_limit=6000,
    random_state=RANDOM_STATE,
)

category_started = time.perf_counter()
category_novelty_bank.fit(
    X_fit.loc[attack_fit_mask, list(ORIGINAL_FEATURES)],
    family_fit[attack_fit_mask],
)
category_novelty_fit_seconds = time.perf_counter() - category_started

category_novelty_state = category_novelty_bank.state()
display(pd.DataFrame(category_novelty_state).T)
print(f"Model 3 bank fit time: {category_novelty_fit_seconds:.2f} seconds")


,algorithm,threshold,fit_rows,reference_rows,encoded_dimensions
DoS,OneClassSVM,11.329086,6000,5151,103
Probe,IsolationForest,-0.071134,5206,1302,111
R2L,OneClassSVM,1.39855,427,107,51
U2R,OneClassSVM,0.175978,28,8,46


Model 3 bank fit time: 0.70 seconds


## 16. Learn route-specific Stage‑2 policy thresholds

These thresholds limit how many known validation attacks can be rejected as unknown or sent to review for each Stage‑1 route. The external test labels are not used here.

In [16]:
policy_diagnostics = stage2_open_router.fit_policy(
    X_validation.loc[:, list(ORIGINAL_FEATURES)],
    y_validation_binary_stage2,
    family_validation,
    stage1_validation,
)

display(pd.DataFrame(policy_diagnostics.route_thresholds).T)
display(pd.DataFrame({
    "all_validation_rows": policy_diagnostics.validation_route_counts,
    "known_validation_attacks": policy_diagnostics.validation_known_attack_counts,
}).fillna(0).astype(int))


,route,unknown_score_threshold,calibration_sample_count,source,allowed_known_rejection_rate,diagnostic_min_confidence,diagnostic_min_margin,diagnostic_max_entropy,diagnostic_max_attack_novelty
ROUTE_STAGE_2,ROUTE_STAGE_2,0.294937,7027,route_specific,0.02,0.999503,0.999315,0.003481,0.980171
NOVEL_ANOMALY_CANDIDATE,NOVEL_ANOMALY_CANDIDATE,0.270572,7036,global_known_fallback,0.1,0.999517,0.999344,0.003389,0.899233
REVIEW_DISAGREEMENT,REVIEW_DISAGREEMENT,0.286924,7036,global_known_fallback,0.05,0.999517,0.999344,0.003389,0.953134


,all_validation_rows,known_validation_attacks
NOVEL_ANOMALY_CANDIDATE,4,0
RETURN_NORMAL,8000,7
REVIEW_DISAGREEMENT,78,2
ROUTE_STAGE_2,7035,7027


## 17. Build the complete router and evaluate the external test set

The global open-set router first decides whether a known category can be accepted. Model 3 then verifies that the record resembles the predicted family. If category novelty rejects an otherwise accepted prediction, the final decision becomes `REVIEW_REQUIRED` rather than forcing a potentially incorrect family.

In [17]:
complete_stage2_router = CompleteStage2Router(
    open_set_router=stage2_open_router,
    category_novelty_bank=category_novelty_bank,
    model_version=STAGE2_MODEL_VERSION,
    category_novelty_version=STAGE2_CATEGORY_NOVELTY_VERSION,
)

stage2_test_output = complete_stage2_router.predict(
    X_test.loc[:, list(ORIGINAL_FEATURES)],
    stage1_test,
)

stage2_metrics, stage2_route_metrics = evaluate_stage2(
    stage2_test_output,
    y_test_binary_stage2,
    family_test_stage2,
    ATTACK_CATEGORIES,
)
accepted_report = stage2_classification_report(
    stage2_test_output,
    family_test_stage2,
)

accepted_mask = stage2_test_output["stage2_decision"].str.startswith("KNOWN_ATTACK_")
predicted_accepted_category = stage2_test_output.loc[
    accepted_mask, "stage2_decision"
].str.replace("KNOWN_ATTACK_", "", regex=False)
accepted_confusion = pd.crosstab(
    pd.Series(family_test_stage2, index=X_test.index).loc[accepted_mask],
    predicted_accepted_category,
    rownames=["true_attack_category"],
    colnames=["accepted_prediction"],
)

category_confirmation_metrics = pd.DataFrame(
    [
        {
            "predicted_category": category,
            "algorithm": CATEGORY_NOVELTY_ALGORITHMS[category],
            "rows_scored": int((stage2_test_output["predicted_known_category"] == category).sum()),
            "category_novelty_rejection_rate": float(
                stage2_test_output.loc[
                    stage2_test_output["predicted_known_category"] == category,
                    "category_novelty_is_novel",
                ].mean()
            ),
        }
        for category in ATTACK_CATEGORIES
    ]
)

truth_stage2 = pd.Series(family_test_stage2, index=X_test.index)
family_final_rows = []
for category in ATTACK_CATEGORIES:
    mask = truth_stage2.eq(category)
    category_decisions = stage2_test_output.loc[mask, "stage2_decision"].astype(str)
    correct_decision = f"KNOWN_ATTACK_{category}"
    family_final_rows.append(
        {
            "attack_category": category,
            "support": int(mask.sum()),
            "correct_known_category_rate": float(category_decisions.eq(correct_decision).mean()),
            "any_known_category_acceptance_rate": float(category_decisions.str.startswith("KNOWN_ATTACK_").mean()),
            "review_rate": float(category_decisions.eq("REVIEW_REQUIRED").mean()),
            "false_unknown_rate": float(category_decisions.eq("UNKNOWN_ATTACK_CANDIDATE").mean()),
            "stage1_false_normal_rate": float(category_decisions.eq("NORMAL_STAGE1_FINAL").mean()),
        }
    )
stage2_family_final_metrics = pd.DataFrame(family_final_rows)

display(pd.DataFrame.from_dict(stage2_metrics, orient="index", columns=["external_test_value"]))
display(stage2_route_metrics)
display(Markdown("### Model 3 scoring by predicted category"))
display(category_confirmation_metrics)
display(Markdown("### Final Stage-2 outcomes by true attack family"))
display(stage2_family_final_metrics)
display(accepted_confusion)
display(stage2_test_output.head(10))


,external_test_value
record_count,25195.000000
known_attack_count,11726.000000
unknown_attack_count,0.000000
normal_count,13469.000000
known_attack_acceptance_rate,0.966655
known_attack_false_unknown_rate,0.000000
known_attack_review_rate,0.033004
unknown_attack_recall,NaN
normal_unknown_escalation_rate,0.001039
normal_known_attack_false_positive_rate,0.001559


,stage1_route,record_count,true_attack_rate,known_attack_decision_rate,unknown_candidate_rate,review_rate,data_quality_exception_rate
0,NOVEL_ANOMALY_CANDIDATE,2,0.000000,0.000000,0.000000,1.000000,0.0
1,RETURN_NORMAL,13343,0.000300,0.000000,0.000000,0.000000,0.0
2,REVIEW_DISAGREEMENT,119,0.033613,0.117647,0.117647,0.764706,0.0
3,ROUTE_STAGE_2,11731,0.998892,0.966840,0.000000,0.033160,0.0


### Model 3 scoring by predicted category

,predicted_category,algorithm,rows_scored,category_novelty_rejection_rate
0,DoS,OneClassSVM,17626,0.396857
1,Probe,IsolationForest,4699,0.249415
2,R2L,OneClassSVM,2730,0.067399
3,U2R,OneClassSVM,140,0.342857


### Final Stage-2 outcomes by true attack family

,attack_category,support,correct_known_category_rate,any_known_category_acceptance_rate,review_rate,false_unknown_rate,stage1_false_normal_rate
0,DoS,9105,0.975618,0.975618,0.024382,0.0,0.000000
1,Probe,2389,0.948514,0.948514,0.050649,0.0,0.000837
2,R2L,226,0.823009,0.823009,0.168142,0.0,0.008850
3,U2R,6,0.000000,0.000000,1.000000,0.0,0.000000


accepted_prediction,DoS,Probe,R2L
true_attack_category,,,
DoS,8883,0,0
Normal,2,15,4
Probe,0,2266,0
R2L,0,0,186


,stage1_route,xgb_attack_probability,normal_novelty_percentile,predicted_known_category,category_confidence,category_second_probability,category_margin,category_entropy,category_probability_DoS,category_probability_Probe,...,stage2_reason,closest_known_category,review_priority,category_novelty_algorithm,category_novelty_score,category_novelty_percentile,category_novelty_threshold,category_novelty_is_novel,stage2_model_version,category_novelty_version
77971,ROUTE_STAGE_2,0.999522,0.999629,DoS,0.999618,0.000173,0.999445,0.002714,0.999618,0.000046,...,Strong Stage-1 attack evidence and acceptable ...,DoS,low,OneClassSVM,-0.365148,0.875388,11.329086,False,stage2-router-v1,stage2-category-novelty-v1
28221,RETURN_NORMAL,0.001061,0.066436,DoS,0.996480,0.001908,0.994572,0.019394,0.996480,0.001908,...,Stage 1 already finalized the record as Normal.,DoS,low,OneClassSVM,16.034775,0.996118,11.329086,True,stage2-router-v1,stage2-category-novelty-v1
18104,ROUTE_STAGE_2,0.999522,0.999505,DoS,0.999618,0.000173,0.999445,0.002714,0.999618,0.000046,...,Strong Stage-1 attack evidence and acceptable ...,DoS,low,OneClassSVM,-2.976800,0.032415,11.329086,False,stage2-router-v1,stage2-category-novelty-v1
11281,ROUTE_STAGE_2,0.999522,0.999876,DoS,0.999618,0.000173,0.999445,0.002714,0.999618,0.000046,...,Strong Stage-1 attack evidence and acceptable ...,DoS,low,OneClassSVM,-0.444511,0.857531,11.329086,False,stage2-router-v1,stage2-category-novelty-v1
8545,RETURN_NORMAL,0.001060,0.094643,DoS,0.996367,0.001914,0.994453,0.019950,0.996367,0.001914,...,Stage 1 already finalized the record as Normal.,DoS,low,OneClassSVM,15.884497,0.995924,11.329086,True,stage2-router-v1,stage2-category-novelty-v1
51714,RETURN_NORMAL,0.001060,0.563034,DoS,0.798394,0.200450,0.597943,0.368261,0.798394,0.200450,...,Stage 1 already finalized the record as Normal.,DoS,low,OneClassSVM,11.491666,0.990295,11.329086,True,stage2-router-v1,stage2-category-novelty-v1
1696,RETURN_NORMAL,0.001061,0.619325,R2L,0.834807,0.150559,0.684248,0.363824,0.012096,0.150559,...,Stage 1 already finalized the record as Normal.,R2L,low,OneClassSVM,0.805111,0.935185,1.398550,False,stage2-router-v1,stage2-category-novelty-v1
51570,RETURN_NORMAL,0.001060,0.212297,DoS,0.627707,0.360077,0.267630,0.520744,0.627707,0.004402,...,Stage 1 already finalized the record as Normal.,DoS,low,OneClassSVM,17.095923,0.997477,11.329086,True,stage2-router-v1,stage2-category-novelty-v1
34232,ROUTE_STAGE_2,0.999522,0.999505,DoS,0.999618,0.000173,0.999445,0.002714,0.999618,0.000046,...,Strong Stage-1 attack evidence and acceptable ...,DoS,low,OneClassSVM,-2.917512,0.036491,11.329086,False,stage2-router-v1,stage2-category-novelty-v1
21845,ROUTE_STAGE_2,0.999522,0.999505,DoS,0.999618,0.000173,0.999445,0.002714,0.999618,0.000046,...,Strong Stage-1 attack evidence and acceptable ...,DoS,low,OneClassSVM,-2.193291,0.197593,11.329086,False,stage2-router-v1,stage2-category-novelty-v1


## 18. Stage‑2 diagnostic plots

In [18]:
fig_stage2_decisions, ax = plt.subplots(figsize=(10, 4))
decision_by_truth = pd.crosstab(
    pd.Series(np.where(y_test_binary_stage2 == 1, "Attack", "Normal"), name="true_binary"),
    stage2_test_output["stage2_decision_type"].reset_index(drop=True),
    normalize="index",
)
decision_by_truth.plot(kind="bar", stacked=True, ax=ax)
ax.set_ylabel("Fraction within true binary class")
ax.set_title("Final Stage-2 decision types")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
fig_stage2_decisions.tight_layout()
display(fig_stage2_decisions)

fig_stage2_evidence, ax = plt.subplots(figsize=(8, 6))
sample_count = min(6000, len(stage2_test_output))
sample = np.random.default_rng(RANDOM_STATE).choice(
    len(stage2_test_output), size=sample_count, replace=False
)
scatter = ax.scatter(
    stage2_test_output["category_confidence"].to_numpy()[sample],
    stage2_test_output["known_attack_novelty_percentile"].to_numpy()[sample],
    c=y_test_binary_stage2[sample],
    cmap="coolwarm",
    s=9,
    alpha=0.5,
)
ax.set_xlabel("Calibrated Random Forest category confidence")
ax.set_ylabel("Global known-attack novelty percentile")
ax.set_title("Stage-2 classifier confidence versus global novelty")
fig_stage2_evidence.colorbar(scatter, ax=ax, label="True binary label")
fig_stage2_evidence.tight_layout()
display(fig_stage2_evidence)

fig_category_novelty, ax = plt.subplots(figsize=(9, 4))
category_confirmation_metrics.plot(
    x="predicted_category",
    y="category_novelty_rejection_rate",
    kind="bar",
    legend=False,
    ax=ax,
    color="darkorange",
)
ax.set_ylabel("Rejection rate")
ax.set_title("Predicted-category novelty rejection on external test")
ax.tick_params(axis="x", rotation=0)
fig_category_novelty.tight_layout()
display(fig_category_novelty)


<Figure size 1000x400 with 1 Axes>

<Figure size 800x600 with 2 Axes>

<Figure size 900x400 with 1 Axes>

## 19. Versioned Stage‑2 output contract

Each input record produces classifier probabilities and uncertainty, global attack novelty, predicted-category novelty, the inherited Stage‑1 evidence, the final Stage‑2 decision, its reason, review priority, and model versions.

In [19]:
STAGE2_OUTPUT_COLUMNS = [
    "stage1_route",
    "xgb_attack_probability",
    "normal_novelty_percentile",
    "predicted_known_category",
    "category_confidence",
    "category_second_probability",
    "category_margin",
    "category_entropy",
    "known_attack_novelty_percentile",
    "category_novelty_algorithm",
    "category_novelty_score",
    "category_novelty_percentile",
    "category_novelty_threshold",
    "category_novelty_is_novel",
    "unknown_score",
    "stage2_decision",
    "stage2_decision_type",
    "stage2_reason",
    "closest_known_category",
    "review_priority",
    "stage2_model_version",
    "category_novelty_version",
]

stage2_output_example = json.loads(
    stage2_test_output.iloc[0][STAGE2_OUTPUT_COLUMNS].to_json()
)
print(json.dumps(stage2_output_example, indent=2))


{
  "stage1_route": "ROUTE_STAGE_2",
  "xgb_attack_probability": 0.9995218052,
  "normal_novelty_percentile": 0.9996288507,
  "predicted_known_category": "DoS",
  "category_confidence": 0.9996178291,
  "category_second_probability": 0.0001732044,
  "category_margin": 0.9994446247,
  "category_entropy": 0.0027141753,
  "known_attack_novelty_percentile": 0.9310687891,
  "category_novelty_algorithm": "OneClassSVM",
  "category_novelty_score": -0.3651477386,
  "category_novelty_percentile": 0.8753881988,
  "category_novelty_threshold": 11.3290861246,
  "category_novelty_is_novel": false,
  "unknown_score": 0.2800805379,
  "stage2_decision": "KNOWN_ATTACK_DoS",
  "stage2_decision_type": "known_attack",
  "stage2_reason": "Strong Stage-1 attack evidence and acceptable known-attack fit.",
  "closest_known_category": "DoS",
  "review_priority": "low",
  "stage2_model_version": "stage2-router-v1",
  "category_novelty_version": "stage2-category-novelty-v1"
}


## 20. Track the complete Stage‑2 implementation in MLflow

In [20]:
stage2_fit_tracking = X_fit.loc[:, list(ORIGINAL_FEATURES)].copy()
stage2_fit_tracking["binary_target"] = np.where(y_fit_binary_stage2 == 1, "Attack", "Normal")
stage2_fit_tracking["attack_category"] = family_fit
stage2_validation_tracking = X_validation.loc[:, list(ORIGINAL_FEATURES)].copy()
stage2_validation_tracking["binary_target"] = np.where(y_validation_binary_stage2 == 1, "Attack", "Normal")
stage2_validation_tracking["attack_category"] = family_validation
stage2_test_tracking = X_test.loc[:, list(ORIGINAL_FEATURES)].copy()
stage2_test_tracking["binary_target"] = np.where(y_test_binary_stage2 == 1, "Attack", "Normal")
stage2_test_tracking["attack_category"] = family_test_stage2

stage2_fit_dataset = mlflow.data.from_pandas(
    stage2_fit_tracking,
    source=str(DATA_PATH.resolve()),
    targets="attack_category",
    name="hybrid_stage2_component_fit",
)
stage2_validation_dataset = mlflow.data.from_pandas(
    stage2_validation_tracking,
    source=str(DATA_PATH.resolve()),
    targets="attack_category",
    name="hybrid_stage2_policy_validation",
)
stage2_test_dataset = mlflow.data.from_pandas(
    stage2_test_tracking,
    source=str(DATA_PATH.resolve()),
    targets="attack_category",
    name="hybrid_stage2_external_test",
)

with mlflow.start_run(run_name=STAGE2_RUN_NAME) as run:
    mlflow.set_tags(
        {
            "task": "hybrid_stage2_attack_open_set_router",
            "model_1": "calibrated_attack_only_random_forest",
            "model_2": "global_attack_oneclasssvm",
            "model_3": "predicted_category_novelty_bank",
            "stage2_model_version": STAGE2_MODEL_VERSION,
            "category_novelty_version": STAGE2_CATEGORY_NOVELTY_VERSION,
            "external_test_used_for_policy": "false",
        }
    )
    mlflow.log_input(stage2_fit_dataset, context="component_fit")
    mlflow.log_input(stage2_validation_dataset, context="policy_validation")
    mlflow.log_input(stage2_test_dataset, context="external_test")
    mlflow.log_params(
        {
            **{f"random_forest_{key}": value for key, value in ATTACK_RF_PARAMS.items()},
            **{f"global_ocsvm_{key}": value for key, value in GLOBAL_ATTACK_NOVELTY_PARAMS.items()},
            "category_calibration_method": "sigmoid",
            "category_calibration_folds": 5,
            "global_attack_novelty_fit_limit": 6000,
            "global_attack_reference_rows": len(attack_reference_frame),
            "category_novelty_threshold_quantile": 0.99,
            "original_feature_count": len(ORIGINAL_FEATURES),
            "component_fit_seconds": component_fit_seconds,
            "category_novelty_fit_seconds": category_novelty_fit_seconds,
            "random_state": RANDOM_STATE,
            "stage2_model_version": STAGE2_MODEL_VERSION,
            "category_novelty_version": STAGE2_CATEGORY_NOVELTY_VERSION,
        }
    )
    mlflow.log_metrics(
        {
            key: float(value)
            for key, value in stage2_metrics.items()
            if np.isfinite(value)
        }
    )
    mlflow.log_dict(PROVENANCE_RUN_NAMES, "provenance/source_run_names.json")
    mlflow.log_dict(provenance_run_ids, "provenance/source_run_ids.json")
    mlflow.log_dict(policy_diagnostics.to_dict(), "policy/fit_diagnostics.json")
    mlflow.log_dict(stage2_open_router.component_state(), "metadata/component_state.json")
    mlflow.log_dict(category_novelty_state, "metadata/category_novelty_bank_state.json")
    mlflow.log_dict(
        {
            "required_input_columns": list(ORIGINAL_FEATURES),
            "required_stage1_columns": list(stage2_open_router.required_stage1_columns),
            "output_columns": STAGE2_OUTPUT_COLUMNS,
            "output_format": "one JSON-compatible record per network flow",
            "known_decisions": [f"KNOWN_ATTACK_{category}" for category in ATTACK_CATEGORIES],
            "open_set_decisions": [
                "UNKNOWN_ATTACK_CANDIDATE",
                "REVIEW_REQUIRED",
                "NORMAL_STAGE1_FINAL",
                "DATA_QUALITY_EXCEPTION",
            ],
        },
        "metadata/input_output_contract.json",
    )
    mlflow.log_dict(
        {
            "python": platform.python_version(),
            "sklearn": sklearn.__version__,
            "mlflow": mlflow.__version__,
            "unknown_ground_truth_available": False,
            "global_novelty_validation": "leave-one-family-out proxy in source experiment",
            "category_novelty_validation": "other-known-family rejection proxy in source experiments",
            "u2r_category_model_low_confidence": True,
        },
        "metadata/run_metadata.json",
    )
    mlflow.log_table(stage2_route_metrics, "evaluation/metrics_by_stage1_route.json")
    mlflow.log_table(category_confirmation_metrics, "evaluation/category_confirmation_metrics.json")
    mlflow.log_table(stage2_family_final_metrics, "evaluation/final_outcomes_by_attack_family.json")
    mlflow.log_table(accepted_confusion.reset_index(), "evaluation/accepted_category_confusion.json")
    mlflow.log_dict(accepted_report, "evaluation/accepted_category_classification_report.json")
    mlflow.log_table(stage2_test_output.head(1000), "examples/stage2_output_sample.json")
    mlflow.log_dict(stage2_output_example, "examples/stage2_output_example.json")
    mlflow.log_figure(fig_stage2_decisions, "plots/stage2_decision_distribution.png")
    mlflow.log_figure(fig_stage2_evidence, "plots/confidence_vs_global_novelty.png")
    mlflow.log_figure(fig_category_novelty, "plots/category_novelty_rejection.png")

    with tempfile.TemporaryDirectory() as temporary_directory:
        router_artifact = Path(temporary_directory) / "complete_stage2_router.joblib"
        policy_artifact = Path(temporary_directory) / "stage2_policy.json"
        complete_stage2_router.save(router_artifact)
        stage2_open_router.export_policy(policy_artifact)
        mlflow.log_artifact(str(router_artifact), artifact_path="router")
        mlflow.log_artifact(str(policy_artifact), artifact_path="router")

    stage2_run_id = run.info.run_id

for figure in [fig_stage2_decisions, fig_stage2_evidence, fig_category_novelty]:
    plt.close(figure)

print("Finished Stage-2 MLflow run:", stage2_run_id)


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Finished Stage-2 MLflow run: 21a3a7a638014352bbc52678c02f79e9


## 21. Stage‑2 measured conclusion

In [21]:
display(
    Markdown(
        f"""### Complete Stage‑2 result

The calibrated attack-only Random Forest, global One-Class SVM novelty model, and predicted-category novelty bank are now integrated into one versioned router.

- Known-attack acceptance: **{stage2_metrics['known_attack_acceptance_rate']:.2%}**
- Category accuracy when accepted: **{stage2_metrics['known_category_accuracy_when_accepted']:.4f}**
- Category macro-F1 when accepted: **{stage2_metrics['known_category_macro_f1_when_accepted']:.4f}**
- Known-attack false-unknown rate: **{stage2_metrics['known_attack_false_unknown_rate']:.2%}**
- Known-attack review rate: **{stage2_metrics['known_attack_review_rate']:.2%}**
- Overall review rate: **{stage2_metrics['overall_review_rate']:.2%}**

Model 3 acts as a safety confirmation layer: an accepted global known-attack prediction is changed to `REVIEW_REQUIRED` when the selected family-specific novelty model rejects it. The `stage2_family_final_metrics` table must be used with aggregate accuracy because it shows coverage and review rates for every family. U2R remains low-confidence because its source experiment has very limited support.

This test set contains only known attack families, so `unknown_attack_recall` is not measurable here. Use leave-one-family-out, temporally newer attacks, or an external unknown-attack dataset before treating `UNKNOWN_ATTACK_CANDIDATE` as production-validated.

**MLflow run:** `{stage2_run_id}`"""
    )
)


### Complete Stage‑2 result

The calibrated attack-only Random Forest, global One-Class SVM novelty model, and predicted-category novelty bank are now integrated into one versioned router.

- Known-attack acceptance: **96.67%**
- Category accuracy when accepted: **1.0000**
- Category macro-F1 when accepted: **1.0000**
- Known-attack false-unknown rate: **0.00%**
- Known-attack review rate: **3.30%**
- Overall review rate: **1.91%**

Model 3 acts as a safety confirmation layer: an accepted global known-attack prediction is changed to `REVIEW_REQUIRED` when the selected family-specific novelty model rejects it. The `stage2_family_final_metrics` table must be used with aggregate accuracy because it shows coverage and review rates for every family. U2R remains low-confidence because its source experiment has very limited support.

This test set contains only known attack families, so `unknown_attack_recall` is not measurable here. Use leave-one-family-out, temporally newer attacks, or an external unknown-attack dataset before treating `UNKNOWN_ATTACK_CANDIDATE` as production-validated.

**MLflow run:** `21a3a7a638014352bbc52678c02f79e9`